In [1]:
from pathlib import Path

import pandas as pd

In [2]:
pd.set_option("display.max_rows", 500)
pd.set_option("display.precision", 3)

In [3]:
ablations = [
    "masking",
    "augmentation",
    "target_norm",
    "decoders",
    "pos_embed",
    "t_patch_size",
    "flat_baselines",
    "input_space_v3",
    "input_norm",
    "patch_size",
    "target_norm_v2",
]

tables = []
for ablation in ablations:
    paths = sorted(Path(f"../{ablation}/output").rglob("eval_v2/*/eval_table.csv"))
    paths = [p for p in paths if "_old" not in str(p)]
    for path in paths:
        table = pd.read_csv(path)
        table = table.loc[table["split"] == "test"]
        # separate handling for logistic and attn probes
        if path.parts[-2].endswith("_logistic"):
            table = (
                table.query("trial > 0")  # only keep 100 randomized split trials, not fixed splits
                .groupby(["model", "repr", "clf", "dataset"])  # average over splits
                .agg({"bacc": "mean"})
                .reset_index()
                .rename(columns={"bacc": "acc"})  # rename bacc -> acc for convenience
            )
        else:
            table = table.loc[:, ["model", "repr", "clf", "dataset", "acc"]]
        table.insert(0, "name", path.parts[-4])
        table.insert(0, "ablation", ablation)
        tables.append(table)


# hack: add extra results for coordinate normalization in input_norm ablation
ablation = "input_norm"
paths = sorted(Path(f"../{ablation}/output").rglob("eval_v2_1/*/eval_table.csv"))
for path in paths:
    print(f"extra result {path}")
    table = pd.read_csv(path)
    table = table.loc[table["split"] == "test"]
    if path.parts[-2].endswith("_logistic"):
        table = (
            table.query("trial > 0")  # only keep 100 randomized split trials, not fixed splits
            .groupby(["model", "repr", "clf", "dataset"])  # average over splits
            .agg({"bacc": "mean"})
            .reset_index()
            .rename(columns={"bacc": "acc"})  # rename bacc -> acc for convenience
        )
    else:
        table = table.loc[:, ["model", "repr", "clf", "dataset", "acc"]]
    table.insert(0, "name", path.parts[-4] + "_coordeval")
    table.insert(0, "ablation", ablation)
    tables.append(table)

table = pd.concat(tables, ignore_index=True)

print(table.shape)
table.head()

extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/aabc_age__patch__logistic/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/aabc_sex__patch__logistic/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/abide_dx__patch__logistic/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/adhd200_dx__patch__logistic/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/adni_ad_vs_cn__patch__logistic/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/hcpya_task21__patch__attn/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/nsd_cococlip__patch__attn/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_frame/eval_v2_1/ppmi_dx__patch__logistic/eval_table.csv
extra result ../input_norm/output/input_norm/nocoord_global/eval_v2_1/aabc_age__patch__logistic/eval_table.csv
ext

,ablation,name,model,repr,clf,dataset,acc
0,masking,tube2x_mr0.5,flat_mae,patch,logistic,aabc_age,0.488
1,masking,tube2x_mr0.5,flat_mae,patch,logistic,aabc_sex,0.874
2,masking,tube2x_mr0.5,flat_mae,patch,logistic,abide_dx,0.574
3,masking,tube2x_mr0.5,flat_mae,patch,logistic,adhd200_dx,0.575
4,masking,tube2x_mr0.5,flat_mae,patch,logistic,adni_ad_vs_cn,0.628


In [4]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}


trait_datasets = [
    "abide_dx",
    "adhd200_dx",
    "adni_ad_vs_cn",
    "ppmi_dx",
    "aabc_age",
    "aabc_sex",
]
state_datasets = ["hcpya_task21", "nsd_cococlip"]

In [5]:
summary = table.pivot_table(
    values="acc", index=["ablation", "name", "model"], columns=["repr", "clf", "dataset"]
)

columns = [("patch", "logistic", ds) for ds in trait_datasets] + [
    ("patch", "attn", ds) for ds in state_datasets
]
summary = summary.loc[:, columns].droplevel([0, 1], axis=1)
summary

dataset                                                      abide_dx  \
ablation       name                         model                       
augmentation   crop0.5-0.8                  flat_mae            0.547   
               crop0.8-1.0                  flat_mae            0.584   
               jitter0.2                    flat_mae            0.610   
               none                         flat_mae            0.599   
               tr0.8                        flat_mae            0.628   
decoders       attn_reg1                    flat_mae            0.626   
               attn_reg1_pep4               flat_mae            0.620   
               cross_reg1                   flat_mae            0.630   
               cross_reg1_pep4              flat_mae            0.608   
               crossreg_reg1                flat_mae            0.584   
               crossreg_reg16               flat_mae            0.588   
               crossreg_reg1_pep4           flat_mae            0.625   
               crossreg_reg4                flat_mae            0.601   
               crossreg_reg4_pep4           flat_mae            0.612   
input_norm     coord_frame                  flat_mae            0.624   
               coord_global                 flat_mae            0.583   
               coord_none                   flat_mae            0.592   
               nocoord_frame                flat_mae            0.514   
               nocoord_frame_coordeval      flat_mae            0.530   
               nocoord_frame_mni            mni_cortex_mae      0.517   
               nocoord_global               flat_mae            0.516   
               nocoord_global_coordeval     flat_mae            0.505   
               nocoord_global_mni           mni_cortex_mae      0.507   
               nocoord_global_mni_coordeval mni_cortex_mae        NaN   
input_space_v3 flat_lr1e-3_1                flat_mae            0.630   
               flat_lr1e-3_2                flat_mae            0.594   
               flat_lr1e-3_3                flat_mae            0.627   
               flat_lr1e-3_4                flat_mae            0.613   
               flat_lr1e-3_5                flat_mae            0.619   
               flat_lr1e-3_6                flat_mae            0.605   
               flat_lr1e-3_7                flat_mae            0.625   
               flat_lr1e-3_8                flat_mae            0.601   
               mni_cortex_lr1e-3_1          mni_cortex_mae      0.605   
               mni_cortex_lr1e-3_2          mni_cortex_mae      0.616   
               mni_cortex_lr1e-3_3          mni_cortex_mae      0.602   
               mni_cortex_lr1e-3_4          mni_cortex_mae      0.597   
               mni_cortex_lr1e-3_5          mni_cortex_mae      0.606   
               mni_cortex_lr1e-3_6          mni_cortex_mae      0.604   
               mni_cortex_lr1e-3_7          mni_cortex_mae      0.612   
               mni_cortex_lr1e-3_8          mni_cortex_mae      0.592   
               schaefer400_lr3e-4_1         schaefer400_mae     0.608   
               schaefer400_lr3e-4_2         schaefer400_mae     0.631   
               schaefer400_lr3e-4_3         schaefer400_mae     0.620   
               schaefer400_lr3e-4_4         schaefer400_mae     0.611   
               schaefer400_lr3e-4_5         schaefer400_mae     0.631   
               schaefer400_lr3e-4_6         schaefer400_mae     0.619   
               schaefer400_lr3e-4_7         schaefer400_mae     0.619   
               schaefer400_lr3e-4_8         schaefer400_mae     0.623   
masking        tube2x_mr0.5                 flat_mae            0.574   
               tube2x_mr0.75                flat_mae            0.627   
               tube2x_mr0.9                 flat_mae            0.607   
               tube2x_mr0.95                flat_mae            0.599   
               tube_mr0.5                   flat_mae      

In [6]:
baseline = summary.loc[("input_space_v3", slice(None), "flat_mae")]
baseline = pd.DataFrame(
    {"acc": baseline.mean(axis=0), "std": baseline.std(axis=0), "count": baseline.count(axis=0)}
)
baseline

,acc,std,count
dataset,,,
abide_dx,0.614,0.013,8
adhd200_dx,0.592,0.010,8
adni_ad_vs_cn,0.624,0.014,8
ppmi_dx,0.588,0.011,8
aabc_age,0.475,0.016,8
aabc_sex,0.874,0.007,8
hcpya_task21,0.989,0.001,8
nsd_cococlip,0.310,0.007,8


In [7]:
def format_acc(acc, dataset="nsd_cococlip", threshold=3, nobad=False):
    ref_acc = baseline.loc[dataset, "acc"]
    ref_std = baseline.loc[dataset, "std"]
    if pd.isna(acc):
        return "—"
    s = f"{acc * 100:.1f}"
    if acc > ref_acc + threshold * ref_std:
        s = r"\textbf{" + s + r"}"
    elif not nobad and (acc < ref_acc - threshold * ref_std):
        s = r"\badc{" + s + r"}"
    if acc == ref_acc:
        s = r"\gc{" + s + r"}"
    return s

In [8]:
# masking table
masking_summary = summary.loc[("masking", slice(None), "flat_mae")]

configs = [
    "uniform_mr0.5",
    "uniform_mr0.75",
    "uniform_mr0.9",
    "uniform_mr0.95",
    "tube_mr0.5",
    "tube_mr0.75",
    "tube_mr0.9",
    "tube_mr0.95",
    "tube2x_mr0.5",
    "tube2x_mr0.75",
    "tube2x_mr0.9",
    "tube2x_mr0.95",
    "tube_mr0.9_pep4",
    "tube_mr0.9_pep8",
]

masking_summary.loc[configs]

dataset,abide_dx,adhd200_dx,adni_ad_vs_cn,ppmi_dx,aabc_age,aabc_sex,hcpya_task21,nsd_cococlip
name,,,,,,,,
uniform_mr0.5,0.559,0.564,0.614,0.557,0.427,0.824,0.974,0.257
uniform_mr0.75,0.586,0.574,0.612,0.596,0.459,0.867,0.982,0.281
uniform_mr0.9,0.607,0.582,0.642,0.582,0.428,0.856,0.988,0.298
uniform_mr0.95,0.593,0.572,0.612,0.588,0.459,0.869,0.988,0.297
tube_mr0.5,0.579,0.593,0.637,0.585,0.466,0.856,0.975,0.234
tube_mr0.75,0.598,0.563,0.657,0.567,0.434,0.875,0.987,0.299
tube_mr0.9,0.607,0.599,0.623,0.592,0.476,0.871,0.988,0.314
tube_mr0.95,0.622,0.602,0.666,0.595,0.454,0.855,0.984,0.304
tube2x_mr0.5,0.574,0.575,0.628,0.580,0.488,0.874,0.975,0.266


In [9]:
# strategy x ratio grid
mask_ratios = [0.5, 0.75, 0.9, 0.95]
strategies = ["uniform", "tube", "tube2x"]

names = {"tube2x": r"tube (2\x)"}

records = []
for strategy in strategies:
    record = {"strategy": names.get(strategy, strategy)}
    for ratio in mask_ratios:
        label = f"{100 * ratio:.0f}"
        key = f"{strategy}_mr{ratio}"
        try:
            acc = masking_summary.loc[key, "nsd_cococlip"]
            record[label] = format_acc(acc)
        except KeyError:
            record[label] = "-"

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_latex(index=False))

\begin{tabular}{lllll}
\toprule
strategy & 50 & 75 & 90 & 95 \\
\midrule
uniform & \badc{25.7} & \badc{28.1} & 29.8 & 29.7 \\
tube & \badc{23.4} & 29.9 & 31.4 & 30.4 \\
tube (2\x) & \badc{26.6} & 31.1 & 30.6 & \badc{28.7} \\
\bottomrule
\end{tabular}



In [10]:
# augmentation
aug_summary = summary.loc[("augmentation", slice(None), "flat_mae")]
aug_summary.iloc[[3, 2, 1, 0]]

dataset,abide_dx,adhd200_dx,adni_ad_vs_cn,ppmi_dx,aabc_age,aabc_sex,hcpya_task21,nsd_cococlip
name,,,,,,,,
none,0.599,0.606,0.604,0.588,0.466,0.878,0.989,0.304
jitter0.2,0.610,0.596,0.639,0.584,0.492,0.881,0.987,0.297
crop0.8-1.0,0.584,0.591,0.624,0.603,0.475,0.857,0.985,0.289
crop0.5-0.8,0.547,0.576,0.597,0.580,0.438,0.827,0.928,0.244


In [11]:
# single column
names = {
    "none": "none",
    "tr0.8": "TR scale",
    "jitter0.2": "gray jitter",
    "crop0.8-1.0": "crop (weak)",
    "crop0.5-0.8": "crop (strong)",
}

records = []

for key, name in names.items():
    record = {
        "case": name,
        "acc": format_acc(aug_summary.loc[key, "nsd_cococlip"]),
    }

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_latex(index=False))

\begin{tabular}{ll}
\toprule
case & acc \\
\midrule
none & 30.4 \\
TR scale & 31.7 \\
gray jitter & 29.7 \\
crop (weak) & 28.9 \\
crop (strong) & \badc{24.4} \\
\bottomrule
\end{tabular}



In [12]:
# target norm
target_summary = summary.loc[(["target_norm", "target_norm_v2"], slice(None), "flat_mae")]
target_summary = target_summary.droplevel([0, 2])
target_summary

dataset,abide_dx,adhd200_dx,adni_ad_vs_cn,ppmi_dx,aabc_age,aabc_sex,hcpya_task21,nsd_cococlip
name,,,,,,,,
patch,0.585,0.585,0.594,0.566,0.478,0.888,0.988,0.314
pca_nc2,0.633,0.596,0.647,0.584,0.460,0.865,0.988,0.308
pca_nc8,0.584,0.526,0.586,0.531,0.413,0.765,0.931,0.205
none,0.619,0.586,0.653,0.582,0.472,0.883,0.989,0.295
pca_nc2_renorm,0.619,0.590,0.642,0.590,0.487,0.872,0.988,0.321
pca_nc8_renorm,0.573,0.524,0.567,0.567,0.423,0.780,0.945,0.246


In [13]:
names = {
    "none": "pix",
    "patch": "$-$ mean",
    "pca_nc2_renorm": "$-$ pca ($d$=2)",
    "pca_nc8_renorm": "$-$ pca ($d$=8)",
}

records = []
# records = [
#     {
#         "case": "none",
#         "acc": format_acc(baseline.loc["nsd_cococlip", "acc"]),
#     }
# ]

for key, name in names.items():
    record = {
        "case": name,
        "acc": format_acc(target_summary.loc[key, "nsd_cococlip"]),
    }

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_latex(index=False))

\begin{tabular}{ll}
\toprule
case & acc \\
\midrule
pix & 29.5 \\
$-$ mean & 31.4 \\
$-$ pca ($d$=2) & 32.1 \\
$-$ pca ($d$=8) & \badc{24.6} \\
\bottomrule
\end{tabular}



In [14]:
# t patch size
pt_summary = summary.loc[("t_patch_size", slice(None), "flat_mae")]

configs = [
    "pt-16",
    "pt-8",
    "pt-4",
    "pt-2",
    "pt-1",
]
pt_summary.loc[configs]

dataset,abide_dx,adhd200_dx,adni_ad_vs_cn,ppmi_dx,aabc_age,aabc_sex,hcpya_task21,nsd_cococlip
name,,,,,,,,
pt-16,0.639,0.596,0.614,0.598,0.467,0.849,0.979,0.276
pt-8,0.625,0.574,0.637,0.582,0.476,0.864,0.987,0.283
pt-4,0.614,0.584,0.629,0.585,0.495,0.881,0.990,0.296
pt-2,0.631,0.596,0.646,0.602,0.471,0.881,0.989,0.329
pt-1,0.629,0.581,0.646,0.576,0.476,0.887,0.990,0.319


In [15]:
patches_per_frame = 364  # fixed num patches per flat map

names = {
    "pt-16": "16",
    "pt-8": "8",
    "pt-4": "4",
    "pt-2": "2",
    "pt-1": "1",
}

records = []
for key, name in names.items():
    record = {
        "$p_t$": name,
        "patches": int(patches_per_frame * 16 / int(name)),
        "acc": format_acc(pt_summary.loc[key, "nsd_cococlip"]),
    }

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_latex(index=False))

\begin{tabular}{lrl}
\toprule
$p_t$ & patches & acc \\
\midrule
16 & 364 & \badc{27.6} \\
8 & 728 & \badc{28.3} \\
4 & 1456 & 29.6 \\
2 & 2912 & 32.9 \\
1 & 5824 & 31.9 \\
\bottomrule
\end{tabular}



In [16]:
names = {
    "pt-16": "16",
    "pt-8": "8",
    "pt-4": "4",
    "pt-2": "2",
    "pt-1": "1",
}

records = []
for key, name in names.items():
    record = {
        "$p_t$": name,
    }

    for ds in trait_datasets[-2:] + state_datasets:
        acc = pt_summary.loc[key, ds]
        record[DATASET_NAMES[ds]] = format_acc(acc, dataset=ds, threshold=10, nobad=True)

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_markdown(index=False))

|   $p_t$ |   HCP-A Age |   HCP-A Sex |   HCP-YA Task21 |   NSD COCO24 |
|--------:|------------:|------------:|----------------:|-------------:|
|      16 |        46.7 |        84.9 |            97.9 |         27.6 |
|       8 |        47.6 |        86.4 |            98.7 |         28.3 |
|       4 |        49.5 |        88.1 |            99   |         29.6 |
|       2 |        47.1 |        88.1 |            98.9 |         32.9 |
|       1 |        47.6 |        88.7 |            99   |         31.9 |


In [17]:
# patch size
ps_summary = summary.loc[("patch_size", slice(None), "flat_mae")]

ps_summary

dataset,abide_dx,adhd200_dx,adni_ad_vs_cn,ppmi_dx,aabc_age,aabc_sex,hcpya_task21,nsd_cococlip
name,,,,,,,,
patch8,0.572,0.558,0.664,0.588,0.426,0.846,0.989,0.294
patch8_mps8,0.594,0.578,0.619,0.598,0.440,0.848,0.985,0.284
patch8_mps8_2,0.564,0.560,0.651,0.594,0.442,0.864,0.987,0.297


In [18]:
pos_summary = summary.loc[("pos_embed", slice(None), "flat_mae")]
pos_summary.iloc[[1, 0, 2]]

dataset,abide_dx,adhd200_dx,adni_ad_vs_cn,ppmi_dx,aabc_age,aabc_sex,hcpya_task21,nsd_cococlip
name,,,,,,,,
sep,0.608,0.597,0.647,0.585,0.484,0.873,0.991,0.321
abs,0.624,0.585,0.624,0.592,0.489,0.881,0.987,0.309
sincos,0.624,0.592,0.661,0.601,0.483,0.881,0.988,0.303


In [19]:
names = {
    "sep": "sep",
    "abs": "abs",
    "sincos": "sincos",
}

records = []
for key, name in names.items():
    record = {
        "case": name,
        "acc": format_acc(pos_summary.loc[key, "nsd_cococlip"]),
    }

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_latex(index=False))

\begin{tabular}{ll}
\toprule
case & acc \\
\midrule
sep & 32.1 \\
abs & 30.9 \\
sincos & 30.3 \\
\bottomrule
\end{tabular}



In [20]:
# input norm
norm_summary = summary.loc[("input_norm", slice(None), slice(None))].droplevel(1)

configs = [
    "coord_frame",
    "coord_global",
    "coord_none",
    "nocoord_frame",
    "nocoord_global",
    "nocoord_frame_mni",
    "nocoord_global_mni",
    "nocoord_frame_coordeval",
    "nocoord_global_coordeval",
    "nocoord_global_mni_coordeval",
]
norm_summary.loc[configs]

dataset,abide_dx,adhd200_dx,adni_ad_vs_cn,ppmi_dx,aabc_age,aabc_sex,hcpya_task21,nsd_cococlip
name,,,,,,,,
coord_frame,0.624,0.588,0.627,0.591,0.481,0.876,0.988,0.314
coord_global,0.583,0.581,0.638,0.599,0.508,0.873,0.977,0.263
coord_none,0.592,0.611,0.647,0.596,0.485,0.852,0.974,0.260
nocoord_frame,0.514,0.564,0.555,0.551,0.384,0.775,0.155,0.054
nocoord_global,0.516,0.583,0.551,0.521,0.402,0.791,0.161,0.055
nocoord_frame_mni,0.517,0.594,0.552,0.530,0.405,0.734,0.107,0.056
nocoord_global_mni,0.507,0.555,0.544,0.494,0.445,0.772,0.160,0.062
nocoord_frame_coordeval,0.530,0.536,0.554,0.532,0.372,0.765,0.744,0.107
nocoord_global_coordeval,0.505,0.530,0.633,0.524,0.408,0.745,0.749,0.095


In [ ]:
names = {
    "coord_frame": (True, True),
    "coord_global": (True, False),
    "nocoord_global": (False, False),
}

records = []
for key, (coord_norm, frame_norm) in names.items():
    record = {
        "global": r"\cmark",
        "coord": r"\cmark" if coord_norm else r"\xmark",
        "frame": r"\cmark" if frame_norm else r"\xmark",
    }

    for col, ds in zip(
        ["Age", "Sex", "Task21", "COCO24"],
        ["aabc_age", "aabc_sex", "hcpya_task21", "nsd_cococlip"],
    ):
        acc_fmt = format_acc(norm_summary.loc[key, ds], dataset=ds)
        if key == "nocoord_global":
            acc = norm_summary.loc[f"{key}_coordeval", ds]
            acc_fmt = f"{acc_fmt} \\tiny{{\\gc{{({100 * acc:.1f})}}}}"
        record[col] = acc_fmt

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_markdown(index=False))
print(df.to_latex(index=False))

| global   | coord   | frame   | Age                            | Sex                            | Task21                         | COCO24                       |
|:---------|:--------|:--------|:-------------------------------|:-------------------------------|:-------------------------------|:-----------------------------|
| \cmark   | \cmark  | \cmark  | 48.1                           | 87.6                           | 98.8                           | 31.4                         |
| \cmark   | \cmark  | \xmark  | 50.8                           | 87.3                           | \badc{97.7}                    | \badc{26.3}                  |
| \cmark   | \xmark  | \xmark  | \badc{40.2} \tiny{\gc{(40.8)}} | \badc{79.1} \tiny{\gc{(74.5)}} | \badc{16.1} \tiny{\gc{(74.9)}} | \badc{5.5} \tiny{\gc{(9.5)}} |
\begin{tabular}{lllllll}
\toprule
global & coord & frame & Age & Sex & Task21 & COCO24 \\
\midrule
\cmark & \cmark & \cmark & 48.1 & 87.6 & 98.8 & 31.4 \\
\cmark & \cmark & \xmark & 50.

In [22]:
names = {
    "coord_frame": (True, True),
    "coord_global": (True, False),
    "nocoord_frame": (False, True),
    "nocoord_global": (False, False),
}

xmark = "✘"
cmark = "✔"

records = []
for key, (coord_norm, frame_norm) in names.items():
    record = {
        "coord": cmark if coord_norm else xmark,
        "frame": cmark if frame_norm else xmark,
        "global": cmark if not frame_norm else xmark,
    }

    for ds in ["aabc_age", "aabc_sex", "hcpya_task21", "nsd_cococlip"]:
        acc_fmt = format_acc(norm_summary.loc[key, ds], dataset=ds, threshold=float("inf"))
        record[DATASET_NAMES[ds]] = acc_fmt

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_markdown(index=False))

| coord   | frame   | global   |   HCP-A Age |   HCP-A Sex |   HCP-YA Task21 |   NSD COCO24 |
|:--------|:--------|:---------|------------:|------------:|----------------:|-------------:|
| ✔       | ✔       | ✘        |        48.1 |        87.6 |            98.8 |         31.4 |
| ✔       | ✘       | ✔        |        50.8 |        87.3 |            97.7 |         26.3 |
| ✘       | ✔       | ✘        |        38.4 |        77.5 |            15.5 |          5.4 |
| ✘       | ✘       | ✔        |        40.2 |        79.1 |            16.1 |          5.5 |


In [23]:
names = {
    "coord_frame": (True, True),
    "nocoord_frame": (False, False),
    "nocoord_frame_coordeval": (False, True),
}

records = []
for key, (coord_pretrain, coord_eval) in names.items():
    record = {
        "coord (pretrain)": cmark if coord_pretrain else xmark,
        "coord (eval)": cmark if coord_eval else xmark,
    }

    for ds in ["aabc_age", "aabc_sex", "hcpya_task21", "nsd_cococlip"]:
        acc_fmt = format_acc(norm_summary.loc[key, ds], dataset=ds, threshold=float("inf"))
        record[DATASET_NAMES[ds]] = acc_fmt

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_markdown(index=False))

| coord (pretrain)   | coord (eval)   |   HCP-A Age |   HCP-A Sex |   HCP-YA Task21 |   NSD COCO24 |
|:-------------------|:---------------|------------:|------------:|----------------:|-------------:|
| ✔                  | ✔              |        48.1 |        87.6 |            98.8 |         31.4 |
| ✘                  | ✘              |        38.4 |        77.5 |            15.5 |          5.4 |
| ✘                  | ✔              |        37.2 |        76.5 |            74.4 |         10.7 |


In [24]:
decoder_summary = (
    table.query("ablation == 'decoders' and model == 'flat_mae'")
    .pivot_table(values="acc", index=["name", "repr", "clf"], columns=["dataset"])
    .loc[:, state_datasets]
    .dropna(axis=1, how="all")
    .dropna(axis=0, how="all")
)

configs = [
    "attn_reg1",
    "cross_reg1",
    "crossreg_reg16",
    "crossreg_reg4",
    "crossreg_reg1",
    "attn_reg1_pep4",
    "cross_reg1_pep4",
    "crossreg_reg4_pep4",
    "crossreg_reg1_pep4",
]
decoder_summary.loc[configs]

dataset                          hcpya_task21  nsd_cococlip
name               repr  clf                               
attn_reg1          patch attn           0.989         0.310
                         linear         0.974         0.129
                   reg   linear         0.960         0.112
cross_reg1         patch attn           0.986         0.298
                         linear         0.967         0.113
                   reg   linear         0.916         0.086
crossreg_reg16     patch attn           0.984         0.258
                         linear         0.905         0.091
                   reg   attn           0.975         0.306
                         linear         0.958         0.109
crossreg_reg4      patch attn           0.986         0.310
                         linear         0.933         0.134
                   reg   attn           0.974         0.264
                         linear         0.963         0.129
crossreg_reg1      patch attn           0.977         0.291
                         linear         0.956         0.219
                   reg   linear         0.975         0.242
attn_reg1_pep4     patch attn           0.991         0.320
                         linear         0.974         0.107
                   reg   linear         0.952         0.106
cross_reg1_pep4    patch attn           0.987         0.313
                         linear         0.972         0.117
                   reg   linear         0.894         0.096
crossreg_reg4_pep4 patch attn           0.986         0.309
                         linear         0.975         0.148
                   reg   linear         0.977         0.201
crossreg_reg1_pep4 patch attn           0.981         0.289
                         linear         0.964         0.188
                   reg   linear         0.975         0.237

In [25]:
# decoding strategy x attn linear
names = {
    "attn_reg1": "self",
    "cross_reg1": "cross",
    "crossreg_reg16": "cross-reg (16)",
    "crossreg_reg4": "cross-reg (4)",
    "crossreg_reg1": "cross-reg (1)",
}

records = []

for key, name in names.items():
    record = {
        "decoder": name,
        "attn": format_acc(decoder_summary.loc[(key, "patch", "attn"), "nsd_cococlip"]),
        "lin": format_acc(decoder_summary.loc[(key, "reg", "linear"), "nsd_cococlip"], nobad=True),
    }

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_latex(index=False))

\begin{tabular}{lll}
\toprule
decoder & attn & lin \\
\midrule
self & 31.0 & 11.2 \\
cross & 29.8 & 8.6 \\
cross-reg (16) & \badc{25.8} & 10.9 \\
cross-reg (4) & 31.0 & 12.9 \\
cross-reg (1) & 29.1 & 24.2 \\
\bottomrule
\end{tabular}



In [26]:
# decoding strategy x with / without decoder masking
names = {
    "attn_reg1": "self",
    "cross_reg1": "cross",
    "crossreg_reg1": "cross-reg (1)",
}

records = []

for key, name in names.items():
    record = {
        "decoder": name,
        "w/o edge mask": format_acc(decoder_summary.loc[(key, "patch", "attn"), "nsd_cococlip"]),
        "w/ edge mask": format_acc(
            decoder_summary.loc[(f"{key}_pep4", "patch", "attn"), "nsd_cococlip"]
        ),
    }

    records.append(record)

df = pd.DataFrame.from_records(records)
print(df.to_latex(index=False))

\begin{tabular}{lll}
\toprule
decoder & w/o edge mask & w/ edge mask \\
\midrule
self & 31.0 & 32.0 \\
cross & 29.8 & 31.3 \\
cross-reg (1) & 29.1 & 28.9 \\
\bottomrule
\end{tabular}

